#### Logic for Basic Property Details Web Scraping

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time
import numpy as np
import pandas as pd
import os
import random

In [ ]:
# Initialize WebDriver

chrome_options = Options()
chrome_options.add_argument("--disable-http2")
chrome_options.add_argument("--incognito")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_argument("--ignore-certificate-errors")
chrome_options.add_argument("--enable-features=NetworkServiceInProcess")
chrome_options.add_argument("--disable-features=NetworkService")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36"
)


driver = webdriver.Chrome(options=chrome_options)
url = "https://www.99acres.com/"
driver.get(url)

# Explicit Wait

wait = WebDriverWait(driver, 5)

def wait_for_page_to_load(driver, wait):
    """Wait until the page is fully loaded."""
    title = driver.title
    try:
        wait.until(lambda d: d.execute_script("return document.readyState") == 'complete')
    except:
        print(f"The webpage \"{title}\" did not get fully loaded.\n")
    else:
        print(f"The webpage \"{title}\" did get fully loaded.\n")

wait_for_page_to_load(driver, wait)


try:
    search_bar = wait.until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="keyword2"]'))
    )
except:
    print("Timeout while locating Search Bar.\n")
else:
    search_bar.send_keys("Gurgaon")
    time.sleep(2)

try:
    valid_option = wait.until(
        EC.element_to_be_clickable((By.XPATH, '//*[@id="0"]'))
    )
except:
    print("Timeout while locating valid search option.\n")
else:
    valid_option.click()
    time.sleep(2)

try:
    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, '//*[@id="searchform_search_btn"]'))
    )
except:
    print("Timeout while clicking on \"Search\" button.\n")
else:
    search_button.click()
    wait_for_page_to_load(driver, wait)

try:
    residential_apartments = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="1"]/span[2]')))
except:
    print("Timeout while clicking on \"Residential Apartment\" option.\n")
else:
    residential_apartments.click()
    time.sleep(2)

try:
    verified = wait.until(
        EC.element_to_be_clickable((By.XPATH, '/html[1]/body[1]/div[1]/div[1]/div[1]/div[4]/div[3]/div[1]/div[3]/section[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[1]/div[3]/span[2]'))
    )
    verified.click()
    time.sleep(2)
except:
    print("Timeout while clicking on \"Verified\" filter.\n")

def handle_popup():
    try:
        popup_button = driver.find_element(By.XPATH, '//*[@id="app"]/div/div[3]/div/div[2]/button')
        popup_button.click()
        time.sleep(2)
    except NoSuchElementException:
        pass

# Navigate pages and extract data

all_properties = []  
page_count = 0

while True:
    page_count += 1
    print(f"\n--- PAGE {page_count} ---")
    print("time:", time.strftime('%Y-%m-%d %H:%M:%S'))
    print("session_id:", driver.session_id)

    # ---- ADD THROTTLING ----
    
    if page_count % 3 == 0:
        print("Throttling: sleeping 3s...")
        time.sleep(3)
    if page_count % 5 == 0:
        print("Throttling: sleeping 10s...")
        time.sleep(3)

    try:
        time.sleep(10)
        next_page_button = driver.find_element(By.XPATH, "//a[normalize-space()='Next Page >']")
    except:
        print(f"\n [COMPLETED] Scraping finished. Total pages navigated: {page_count}.\n")
        break
    else:
        try:
            driver.execute_script("window.scrollBy(0, arguments[0].getBoundingClientRect().top - 100);", next_page_button)
            time.sleep(10)

            main_container = driver.find_element(By.CSS_SELECTOR, 'div[data-label="SEARCH"]')
            property_containers = main_container.find_elements(By.CLASS_NAME, 'tupleNew__contentWrap')
            print(f"Number of Property Found on page {page_count}: {len(property_containers)}")

            for container in property_containers:
                property_data = {}
                try:
                    try:
                        property_data['property_name'] = container.find_element(By.CLASS_NAME, "tupleNew__propType").text
                    except:
                        property_data['property_name'] = np.nan
                    try:
                        property_data['link'] = container.find_element(By.CLASS_NAME, "tupleNew__propertyHeading.ellipsis").get_attribute("href")
                    except:
                        property_data['link'] = np.nan
                    try:
                        property_data['society'] = container.find_element(By.CLASS_NAME, "tupleNew__locationName").text
                    except:
                        property_data['society'] = np.nan
                    try:
                        property_data['price'] = container.find_element(By.CLASS_NAME, "tupleNew__priceValWrap").text
                    except:
                        property_data['price'] = np.nan
                    try:
                        property_data['area'] = container.find_element(By.CSS_SELECTOR, "div.tupleNew__perSqftWrap.ellipsis").text
                    except:
                        property_data['area'] = np.nan
                    try:
                        area1 = container.find_element(By.CLASS_NAME, "tupleNew__area1Type").text
                    except:
                        area1 = ""
                    try:
                        area2 = container.find_element(By.CLASS_NAME, "tupleNew__area2Type").text
                    except:
                        area2 = ""
                    try:
                        area_type = container.find_element(By.CLASS_NAME, "tupleNew__areaType").text
                    except:
                        area_type = ""

                    property_data['areawithtype'] = f"{area1} {area2} {area_type}".strip()
                    all_properties.append(property_data)

                except:
                    print("Not able to extract property details!!")

            next_page_button = wait.until(
                EC.presence_of_element_located((By.XPATH, "//a[normalize-space()='Next Page >']"))
                                            )
            driver.execute_script("arguments[0].click();", next_page_button)
            print("✔ Next page clicked successfully")
            time.sleep(5)

        except Exception as e:
            print("Timeout or error while clicking on Next Page:", e)


The webpage "India Real Estate Property Site - Buy Sell Rent Properties Portal - 99acres.com" did get fully loaded.

The webpage "Property in Gurgaon - Real Estate in Gurgaon" did get fully loaded.


--- PAGE 1 ---
time: 2025-10-29 15:02:04
session_id: f22cbee54a3175921fe4eeff7d0d30dd
Number of Property Found on page 1: 78
✔ Next page clicked successfully

--- PAGE 2 ---
time: 2025-10-29 15:03:28
session_id: f22cbee54a3175921fe4eeff7d0d30dd
Number of Property Found on page 2: 51
✔ Next page clicked successfully

--- PAGE 3 ---
time: 2025-10-29 15:04:37
session_id: f22cbee54a3175921fe4eeff7d0d30dd
Throttling: sleeping 3s...
Number of Property Found on page 3: 26
✔ Next page clicked successfully

--- PAGE 4 ---
time: 2025-10-29 15:05:24
session_id: f22cbee54a3175921fe4eeff7d0d30dd
Number of Property Found on page 4: 26
✔ Next page clicked successfully

--- PAGE 5 ---
time: 2025-10-29 15:06:09
session_id: f22cbee54a3175921fe4eeff7d0d30dd
Throttling: sleeping 10s...
Number of Property Foun

In [16]:
all_properties

[{'property_name': '2 BHK Flat in Sector 112, Gurgaon',
  'link': 'https://www.99acres.com/2-bhk-bedroom-apartment-flat-for-sale-in-tata-gurgaon-gateway-sector-112-gurgaon-1580-sq-ft-spid-L84924514',
  'society': 'Tata Gurgaon Gateway',
  'price': '₹2.4 Cr',
  'area': '₹15,189 /sqft',
  'areawithtype': '1,580 sqft (147 sqm) Super Built-up Area'},
 {'property_name': '3 BHK Flat in Sector 104, Gurgaon',
  'link': 'https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-hero-homes-sector-104-gurgaon-1689-sq-ft-spid-D85625548',
  'society': 'Hero Homes',
  'price': '₹2.38 Cr',
  'area': '₹14,091 /sqft',
  'areawithtype': '1,689 sqft (157 sqm) Super Built-up Area'},
 {'property_name': '3 BHK Flat in Sector 106, Gurgaon',
  'link': 'https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-paras-dews-sector-106-gurgaon-1760-sq-ft-spid-I85922364',
  'society': 'Paras Dews',
  'price': '₹1.95 Cr',
  'area': '₹11,079 /sqft',
  'areawithtype': '1,760 sqft (164 sqm) Super Built-up

In [ ]:
Apartment_df = pd.DataFrame(all_properties)

In [ ]:
Apartment_df.to_csv(r'C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apart_basic_details.csv', index= False)

In [ ]:
residential_apar_basic_details = pd.read_csv(r'C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apart_basic_details.csv')

In [42]:
residential_apar_basic_details.duplicated().sum()

0

In [43]:
residential_apar_basic_details.shape

(2460, 6)

In [44]:
residential_apar_basic_details.isnull().sum()

property_name    0
link             0
society          0
price            0
area             0
areawithtype     0
dtype: int64

### Logic for Detailed Property Page

In [ ]:
# 1. HELPER FUNCTIONS

def wait_for_page_to_load(driver, timeout=15):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )


def get_text(driver, by, value):
    try:
        return driver.find_element(by, value).text.strip()
    except:
        return np.nan


def get_list(driver, css_selector):
    try:
        items = driver.find_elements(By.CSS_SELECTOR, css_selector)
        return [i.text.strip() for i in items if i.text.strip()]
    except:
        return []


# 2. SCRAPE SINGLE PROPERTY PAGE

def scrape_details(driver, link, retries=2):
    """
    Scrapes one property detail page.
    Returns dictionary.
    """

    for _ in range(retries):
        try:
            driver.get(link)
            wait_for_page_to_load(driver)

            data = {
                "link": link,
                "bedrooms": get_text(driver, By.ID, "bedRoomNum"),
                "bathrooms": get_text(driver, By.ID, "bathroomNum"),
                "balcony": get_text(driver, By.ID, "balconyNum"),
                "additional_room": get_text(driver, By.ID, "additionalRooms"),
                "floor_info": get_text(driver, By.ID, "floorNumLabel"),
                "facing": get_text(driver, By.ID, "facingLabel"),
                "property_age": get_text(driver, By.ID, "agePossessionLbl"),
                "property_id": get_text(driver, By.ID, "Prop_Id"),
                "furnishing_details": get_list(driver, "#FurnishDetails ul#features li div"),
                "features": get_list(driver, "div[data-label='FACILITIES'] ul#features li div")
            }

            return data

        except Exception:
            time.sleep(3)

    return {"link": link}


# 3. LOAD INPUT LINKS

input_csv = r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apart_basic_details.csv"
output_file = r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apart_detailed_page.xlsx"

df_links = pd.read_csv(input_csv)
links = df_links["link"].dropna().tolist()
total_links = len(links)

# Load existing output 

if os.path.exists(output_file):
    existing_df = pd.read_excel(output_file)
    processed_links = set(existing_df["link"])
else:
    existing_df = pd.DataFrame()
    processed_links = set()

start = int(input("Enter link number to start from (1 for beginning): ")) - 1


# 4. SETUP DRIVER 

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)


# 5. MAIN LOOP 

BATCH_SIZE = 50

try:
    while start < total_links:
        end = min(start + BATCH_SIZE, total_links)
        print(f"\nScraping links {start + 1} to {end} of {total_links}")

        batch_data = []

        for i in range(start, end):
            link = links[i]

            if link in processed_links:
                continue

            print(f"→ Processing {i + 1}/{total_links}")
            data = scrape_details(driver, link)
            batch_data.append(data)

            time.sleep(random.uniform(2, 4))

        if batch_data:
            new_df = pd.DataFrame(batch_data)

            if not existing_df.empty:
                combined_df = (
                    pd.concat([existing_df, new_df])
                    .drop_duplicates(subset=["link"], keep="last")
                    .reset_index(drop=True)
                )
            else:
                combined_df = new_df

            combined_df.to_excel(output_file, index=False)
            existing_df = combined_df
            processed_links.update(new_df["link"].tolist())

            print(f"Saved data up to link {end}")

        start = end

        if start < total_links:
            print("Cooling down...")
            time.sleep(20)

    print("\nAll links processed successfully!")

finally:
    driver.quit()
    print("Driver closed.")



Scraping links 1 to 50 of 2460
Cooling down...

Scraping links 51 to 100 of 2460
Cooling down...

Scraping links 101 to 150 of 2460
Cooling down...

Scraping links 151 to 200 of 2460
Cooling down...

Scraping links 201 to 250 of 2460
Cooling down...

Scraping links 251 to 300 of 2460
Cooling down...

Scraping links 301 to 350 of 2460
Cooling down...

Scraping links 351 to 400 of 2460
Cooling down...

Scraping links 401 to 450 of 2460
Cooling down...

Scraping links 451 to 500 of 2460
Cooling down...

Scraping links 501 to 550 of 2460
Cooling down...

Scraping links 551 to 600 of 2460
Cooling down...

Scraping links 601 to 650 of 2460
Cooling down...

Scraping links 651 to 700 of 2460
Cooling down...

Scraping links 701 to 750 of 2460
Cooling down...

Scraping links 751 to 800 of 2460
Cooling down...

Scraping links 801 to 850 of 2460
Cooling down...

Scraping links 851 to 900 of 2460
Cooling down...

Scraping links 901 to 950 of 2460
Cooling down...

Scraping links 951 to 1000 of 2460

In [46]:
residential_apar_basic_details

,property_name,link,society,price,area,areawithtype
0,"2 BHK Flat in Sector 112, Gurgaon",https://www.99acres.com/2-bhk-bedroom-apartmen...,Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area"
1,"3 BHK Flat in Sector 37C, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area"
2,"4 BHK Flat in Sector 84, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area"
3,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area"
4,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area"
...,...,...,...,...,...,...
2455,"4 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,Adani M2K Oyster Grande,₹3.6 Cr,"₹11,257 /sqft","3,198 sqft (297 sqm) Super Built-up Area"
2456,"3 BHK Flat in Sector 69, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Tulip Yellow,₹2.33 Cr,"₹13,673 /sqft","1,704 sqft (158 sqm) Super Built-up Area"
2457,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.6 Cr,"₹14,038 /sqft","1,852 sqft (172 sqm) Super Built-up Area"
2458,"3 BHK Flat in Sector 82, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Mapsko Casa Bella,₹2.05 Cr,"₹10,459 /sqft","1,960 sqft (182 sqm) Super Built-up Area"


In [47]:
residential_apar_basic_details['property_id']  = residential_apar_basic_details['link'].str.split("spid-").str.get(1)
residential_apar_basic_details

,property_name,link,society,price,area,areawithtype,property_id
0,"2 BHK Flat in Sector 112, Gurgaon",https://www.99acres.com/2-bhk-bedroom-apartmen...,Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area",L84924514
1,"3 BHK Flat in Sector 37C, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area",W81100115
2,"4 BHK Flat in Sector 84, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area",M80850013
3,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area",Q85232920
4,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area",K85040518
...,...,...,...,...,...,...,...
2455,"4 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,Adani M2K Oyster Grande,₹3.6 Cr,"₹11,257 /sqft","3,198 sqft (297 sqm) Super Built-up Area",G81998138
2456,"3 BHK Flat in Sector 69, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Tulip Yellow,₹2.33 Cr,"₹13,673 /sqft","1,704 sqft (158 sqm) Super Built-up Area",P84081948
2457,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.6 Cr,"₹14,038 /sqft","1,852 sqft (172 sqm) Super Built-up Area",W78586891
2458,"3 BHK Flat in Sector 82, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Mapsko Casa Bella,₹2.05 Cr,"₹10,459 /sqft","1,960 sqft (182 sqm) Super Built-up Area",O24958549


In [ ]:
residential_apar_detailed_page = pd.read_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apart_detailed_page.csv")
residential_apar_detailed_page

,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,property_id,furnishing_details,features,nearby_location
0,https://www.99acres.com/2-bhk-bedroom-apartmen...,2 Bedrooms,2 Bathrooms,2 Balconies,Study Room,5th of 24 Floors,North-East,0 to 1 Year Old,L84924514,"['3 Wardrobe', '1 Water Purifier', '4 Fan', '3...","['Centrally Air Conditioned', 'Water purifier'...","['Metro Hospital, Palam Vihar', 'Mount Carmel ..."
1,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,Ground of 13 Floors,North-West,1 to 5 Year Old,W81100115,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '1 Ge...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Elan Miracle Mall', 'Holiday Inn Gurugram Se..."
2,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,4 Bathrooms,2 Balconies,"Servant Room,Store Room",3rd of 4 Floors,North-East,0 to 1 Year Old,M80850013,"['4 Wardrobe', '8 Fan', '1 Exhaust Fan', '15 L...","['Lift(s)', 'High Ceiling Height', 'Separate e...","['AapnoGhar', 'Modern Public School-kherki Dau..."
3,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,12nd of 26 Floors,North-East,1 to 5 Year Old,Q85232920,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Hyatt Place Gurgaon Udyog Vihar', 'The Espla..."
4,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,7th of 14 Floors,North-East,1 to 5 Year Old,K85040518,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Power Back-up', 'Fe...","['Sunrise University', 'Country Inn & Suites b..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2455,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,5 Bathrooms,3+ Balconies,Servant Room,5th of 24 Floors,North-West,1 to 5 Year Old,G81998138,"['5 Fan', '1 Exhaust Fan', '4 Geyser', '25 Lig...","['Centrally Air Conditioned', 'Water purifier'...","['Gurgaon', 'SGT University', 'Indira Gandhi A..."
2456,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3+ Balconies,Pooja Room,3rd of 26 Floors,North-East,0 to 1 Year Old,P84081948,NaN,"['Centrally Air Conditioned', 'Water purifier'...","['Hyatt Regency Gurugram', 'De Adventure Park'..."
2457,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,14th of 26 Floors,North-East,0 to 1 Year Old,W78586891,"['3 Wardrobe', '3 Fan', '3 Light', '1 Modular ...","['Security / Fire Alarm', 'Intercom Facility',...","['IFFCO Chowk Metro Station', 'The Esplanade M..."
2458,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,6th of 20 Floors,North-West,1 to 5 Year Old,O24958549,"['4 Wardrobe', '7 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['SkyJumper Trampoline Park', 'Pataudi Road', ..."


In [50]:
residential_apar_detailed_page.duplicated().sum()

0

In [51]:
residential_apar_detailed_page.shape

(2460, 12)

In [52]:
residential_apartment_merge = residential_apar_basic_details.merge(residential_apar_detailed_page, on='property_id',how= 'inner')
residential_apartment_merge

,property_name,link_x,society,price,area,areawithtype,property_id,link_y,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,"2 BHK Flat in Sector 112, Gurgaon",https://www.99acres.com/2-bhk-bedroom-apartmen...,Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area",L84924514,https://www.99acres.com/2-bhk-bedroom-apartmen...,2 Bedrooms,2 Bathrooms,2 Balconies,Study Room,5th of 24 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '1 Water Purifier', '4 Fan', '3...","['Centrally Air Conditioned', 'Water purifier'...","['Metro Hospital, Palam Vihar', 'Mount Carmel ..."
1,"3 BHK Flat in Sector 37C, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area",W81100115,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,Ground of 13 Floors,North-West,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '1 Ge...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Elan Miracle Mall', 'Holiday Inn Gurugram Se..."
2,"4 BHK Flat in Sector 84, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area",M80850013,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,4 Bathrooms,2 Balconies,"Servant Room,Store Room",3rd of 4 Floors,North-East,0 to 1 Year Old,"['4 Wardrobe', '8 Fan', '1 Exhaust Fan', '15 L...","['Lift(s)', 'High Ceiling Height', 'Separate e...","['AapnoGhar', 'Modern Public School-kherki Dau..."
3,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area",Q85232920,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,12nd of 26 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Hyatt Place Gurgaon Udyog Vihar', 'The Espla..."
4,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area",K85040518,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,7th of 14 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Power Back-up', 'Fe...","['Sunrise University', 'Country Inn & Suites b..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2455,"4 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/4-bhk-bedroom-apartmen...,Adani M2K Oyster Grande,₹3.6 Cr,"₹11,257 /sqft","3,198 sqft (297 sqm) Super Built-up Area",G81998138,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,5 Bathrooms,3+ Balconies,Servant Room,5th of 24 Floors,North-West,1 to 5 Year Old,"['5 Fan', '1 Exhaust Fan', '4 Geyser', '25 Lig...","['Centrally Air Conditioned', 'Water purifier'...","['Gurgaon', 'SGT University', 'Indira Gandhi A..."
2456,"3 BHK Flat in Sector 69, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Tulip Yellow,₹2.33 Cr,"₹13,673 /sqft","1,704 sqft (158 sqm) Super Built-up Area",P84081948,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3+ Balconies,Pooja Room,3rd of 26 Floors,North-East,0 to 1 Year Old,NaN,"['Centrally Air Conditioned', 'Water purifier'...","['Hyatt Regency Gurugram', 'De Adventure Park'..."
2457,"3 BHK Flat in Sector 102, Gurgaon",https://www.99acres.com/3-bhk-bedroom-apartmen...,Shapoorji Pallonji Joyville Gurugram,₹2.6 Cr,"₹14,038 /sqft","1,852 sqft (172 sqm) Super Built-up Area",W78586891,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,14th of 26 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '3 Fan', '3 Light', '1 Mod

In [53]:
residential_apartment_merge.isnull().sum()

property_name           0
link_x                  0
society                 0
price                   0
area                    0
areawithtype            0
property_id             0
link_y                  0
bedrooms                0
bathrooms               0
balcony                 0
additional_room       599
floor_info              0
facing                106
property_age            4
furnishing_details    403
features                3
nearby_location         0
dtype: int64

In [54]:
residential_apartment = residential_apartment_merge.drop(columns = ['link_x']).rename(columns ={'link_y':'link'})
residential_apartment

,property_name,society,price,area,areawithtype,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,"2 BHK Flat in Sector 112, Gurgaon",Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area",L84924514,https://www.99acres.com/2-bhk-bedroom-apartmen...,2 Bedrooms,2 Bathrooms,2 Balconies,Study Room,5th of 24 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '1 Water Purifier', '4 Fan', '3...","['Centrally Air Conditioned', 'Water purifier'...","['Metro Hospital, Palam Vihar', 'Mount Carmel ..."
1,"3 BHK Flat in Sector 37C, Gurgaon",Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area",W81100115,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,Ground of 13 Floors,North-West,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '1 Ge...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Elan Miracle Mall', 'Holiday Inn Gurugram Se..."
2,"4 BHK Flat in Sector 84, Gurgaon",SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area",M80850013,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,4 Bathrooms,2 Balconies,"Servant Room,Store Room",3rd of 4 Floors,North-East,0 to 1 Year Old,"['4 Wardrobe', '8 Fan', '1 Exhaust Fan', '15 L...","['Lift(s)', 'High Ceiling Height', 'Separate e...","['AapnoGhar', 'Modern Public School-kherki Dau..."
3,"3 BHK Flat in Sector 102, Gurgaon",Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area",Q85232920,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,12nd of 26 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Hyatt Place Gurgaon Udyog Vihar', 'The Espla..."
4,"3 BHK Flat in Sector 102, Gurgaon",Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area",K85040518,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,7th of 14 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Power Back-up', 'Fe...","['Sunrise University', 'Country Inn & Suites b..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2455,"4 BHK Flat in Sector 102, Gurgaon",Adani M2K Oyster Grande,₹3.6 Cr,"₹11,257 /sqft","3,198 sqft (297 sqm) Super Built-up Area",G81998138,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,5 Bathrooms,3+ Balconies,Servant Room,5th of 24 Floors,North-West,1 to 5 Year Old,"['5 Fan', '1 Exhaust Fan', '4 Geyser', '25 Lig...","['Centrally Air Conditioned', 'Water purifier'...","['Gurgaon', 'SGT University', 'Indira Gandhi A..."
2456,"3 BHK Flat in Sector 69, Gurgaon",Tulip Yellow,₹2.33 Cr,"₹13,673 /sqft","1,704 sqft (158 sqm) Super Built-up Area",P84081948,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3+ Balconies,Pooja Room,3rd of 26 Floors,North-East,0 to 1 Year Old,NaN,"['Centrally Air Conditioned', 'Water purifier'...","['Hyatt Regency Gurugram', 'De Adventure Park'..."
2457,"3 BHK Flat in Sector 102, Gurgaon",Shapoorji Pallonji Joyville Gurugram,₹2.6 Cr,"₹14,038 /sqft","1,852 sqft (172 sqm) Super Built-up Area",W78586891,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,14th of 26 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '3 Fan', '3 Light', '1 Modular ...","['Security / Fire Alarm', 'Intercom Facility',...","['IFFCO Chowk Metro Station', 'The Esplanade M..."
2458,"3 BHK Flat in Sector 82, Gurgaon",Mapsko Casa Bella,₹2.05 Cr,"₹10,459 /sqft","1,960 sqft (182 sqm) Super Built-up Area",O24958549,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,6th of 20 Floors,North-West,1 to 5 Year Old,"['4 Wardrobe', '7 Fan', '1 Ex

In [55]:
residential_apartment.columns = residential_apartment.columns.str.lower()

In [56]:
residential_apartment.head()

,property_name,society,price,area,areawithtype,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,"2 BHK Flat in Sector 112, Gurgaon",Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area",L84924514,https://www.99acres.com/2-bhk-bedroom-apartmen...,2 Bedrooms,2 Bathrooms,2 Balconies,Study Room,5th of 24 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '1 Water Purifier', '4 Fan', '3...","['Centrally Air Conditioned', 'Water purifier'...","['Metro Hospital, Palam Vihar', 'Mount Carmel ..."
1,"3 BHK Flat in Sector 37C, Gurgaon",Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area",W81100115,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,Ground of 13 Floors,North-West,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '1 Ge...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Elan Miracle Mall', 'Holiday Inn Gurugram Se..."
2,"4 BHK Flat in Sector 84, Gurgaon",SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area",M80850013,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,4 Bathrooms,2 Balconies,"Servant Room,Store Room",3rd of 4 Floors,North-East,0 to 1 Year Old,"['4 Wardrobe', '8 Fan', '1 Exhaust Fan', '15 L...","['Lift(s)', 'High Ceiling Height', 'Separate e...","['AapnoGhar', 'Modern Public School-kherki Dau..."
3,"3 BHK Flat in Sector 102, Gurgaon",Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area",Q85232920,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,12nd of 26 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Hyatt Place Gurgaon Udyog Vihar', 'The Espla..."
4,"3 BHK Flat in Sector 102, Gurgaon",Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area",K85040518,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,7th of 14 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Power Back-up', 'Fe...","['Sunrise University', 'Country Inn & Suites b..."


In [57]:
residential_apartment.shape

(2460, 17)

In [ ]:
residential_apartment.to_csv(r"C:\Users\Jay Patel\Campusx\ml_projects\PropNavigator\data\web_scraping\residential_apartment.csv", index= False)

In [59]:
residential_apartment.head()

,property_name,society,price,area,areawithtype,property_id,link,bedrooms,bathrooms,balcony,additional_room,floor_info,facing,property_age,furnishing_details,features,nearby_location
0,"2 BHK Flat in Sector 112, Gurgaon",Tata Gurgaon Gateway,₹2.5 Cr,"₹15,822 /sqft","1,580 sqft (147 sqm) Super Built-up Area",L84924514,https://www.99acres.com/2-bhk-bedroom-apartmen...,2 Bedrooms,2 Bathrooms,2 Balconies,Study Room,5th of 24 Floors,North-East,0 to 1 Year Old,"['3 Wardrobe', '1 Water Purifier', '4 Fan', '3...","['Centrally Air Conditioned', 'Water purifier'...","['Metro Hospital, Palam Vihar', 'Mount Carmel ..."
1,"3 BHK Flat in Sector 37C, Gurgaon",Ramprastha The View,₹1.29 Cr,"₹10,254 /sqft","1,565 sqft (145 sqm) Super Built-up Area",W81100115,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,Ground of 13 Floors,North-West,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '1 Ge...","['Power Back-up', 'Intercom Facility', 'Lift(s...","['Elan Miracle Mall', 'Holiday Inn Gurugram Se..."
2,"4 BHK Flat in Sector 84, Gurgaon",SS Linden Floors,₹2.9 Cr,"₹10,653 /sqft","2,722 sqft (253 sqm) Super Built-up Area",M80850013,https://www.99acres.com/4-bhk-bedroom-apartmen...,4 Bedrooms,4 Bathrooms,2 Balconies,"Servant Room,Store Room",3rd of 4 Floors,North-East,0 to 1 Year Old,"['4 Wardrobe', '8 Fan', '1 Exhaust Fan', '15 L...","['Lift(s)', 'High Ceiling Height', 'Separate e...","['AapnoGhar', 'Modern Public School-kherki Dau..."
3,"3 BHK Flat in Sector 102, Gurgaon",Shapoorji Pallonji Joyville Gurugram,₹2.65 Cr,"₹20,075 /sqft","1,852 sqft (172 sqm) Super Built-up Area",Q85232920,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,3 Bathrooms,3 Balconies,NaN,12nd of 26 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Feng Shui / Vaastu ...","['Hyatt Place Gurgaon Udyog Vihar', 'The Espla..."
4,"3 BHK Flat in Sector 102, Gurgaon",Emaar Imperial Gardens,₹2.4 Cr,"₹11,851 /sqft","2,025 sqft (188 sqm) Super Built-up Area",K85040518,https://www.99acres.com/3-bhk-bedroom-apartmen...,3 Bedrooms,4 Bathrooms,3+ Balconies,Servant Room,7th of 14 Floors,North-East,1 to 5 Year Old,"['3 Wardrobe', '5 Fan', '1 Exhaust Fan', '3 Ge...","['Security / Fire Alarm', 'Power Back-up', 'Fe...","['Sunrise University', 'Country Inn & Suites b..."
